In [4]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize


df = pd.read_excel("cleaneddata.xlsx")
df.columns = df.columns.str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["cusip","date"])

# numeric cleanup
for col in ["spread","price","sduration","coupon","ytm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["date","spread","price","sduration"])

# restrict sample window
df = df[df["date"].between("2017-01-01","2019-12-31")]
df["ym"] = df["date"].dt.to_period("M")

price_col = "price"

first_day = (
    df.sort_values("date")
      .groupby(["cusip","ym"])
      .first()
      .reset_index()
)

first_day["next_price"] = first_day.groupby("cusip")[price_col].shift(-1)
first_day["next_ym"]    = first_day.groupby("cusip")["ym"].shift(-1)

first_day = first_day.dropna(subset=["next_price","next_ym"])

first_day = first_day[
    first_day["ym"].between("2017-01","2019-12") &
    first_day["next_ym"].between("2017-01","2019-12")
]

# price return
first_day["price_ret"] = (
    (first_day["next_price"] - first_day[price_col]) /
     first_day[price_col]
)

# carry = coupon%/12 / price
first_day["carry"] = ((first_day["coupon"] / 100) / 12) / first_day[price_col]

# total return
first_day["ret"] = first_day["price_ret"] + first_day["carry"]

# DTS = spread × spread duration
first_day["dts"] = first_day["spread"] * first_day["sduration"]

# return matrix
ret_pivot = (
    first_day.pivot(index="ym", columns="cusip", values="ret")
    .sort_index()
)

In [5]:
# allows for long short to build duration neutral
def optimize_duration_neutral(mu, cov, sdur, w_max=0.10):
    """
    Long-short duration-neutral optimizer
    """
    n = len(mu)
    w0 = np.ones(n) / n

    def neg_sharpe(w):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(cov, w)))
        if vol <= 0:
            return 1e9
        return -(ret / vol)

    bounds = [(-w_max, w_max)] * n

    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w: np.dot(w, sdur)}
    ]

    res = minimize(
        neg_sharpe,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 1000, "ftol": 1e-9}
    )

    return res.x

In [6]:

dn_low  = []
dn_mid  = []
dn_high = []

months = sorted(first_day["ym"].unique())

for idx in range(3, len(months)):
    ym = months[idx]

    # trailing 3 months for covariance
    window = months[idx-3:idx]
    hist = ret_pivot.loc[window]

    # current month bonds
    month_df = first_day[first_day["ym"] == ym].copy()
    month_df = month_df.sort_values("dts")
    n = len(month_df)
    if n < 6:
        continue

    # split into 3 equal buckets
    k = n // 3
    low  = month_df.iloc[:k].copy()
    mid  = month_df.iloc[k:2*k].copy()
    high = month_df.iloc[2*k:3*k].copy()

    def prep(bucket):
        cus = list(bucket["cusip"])
        sub = hist[cus].dropna(axis=1, how="any")
        if sub.shape[1] < 2:
            return None, None, None

        cusips = list(sub.columns)
        bucket2 = bucket.set_index("cusip").loc[cusips].reset_index()

        mu  = sub.mean().values
        cov = np.cov(sub.T)

        return bucket2, mu, cov

    low_b,  mu_low,  cov_low  = prep(low)
    mid_b,  mu_mid,  cov_mid  = prep(mid)
    high_b, mu_high, cov_high = prep(high)

    if low_b is None or mid_b is None or high_b is None:
        continue

    # spread-duration arrays
    sdur_low  = low_b["sduration"].values
    sdur_mid  = mid_b["sduration"].values
    sdur_high = high_b["sduration"].values

    # duration-neutral weights (UNHEDGED)
    w_low  = optimize_duration_neutral(mu_low,  cov_low,  sdur_low)
    w_mid  = optimize_duration_neutral(mu_mid,  cov_mid,  sdur_mid)
    w_high = optimize_duration_neutral(mu_high, cov_high, sdur_high)

    # realized return (no hedge)
    ret_low  = np.dot(w_low,  low_b["ret"].values)
    ret_mid  = np.dot(w_mid,  mid_b["ret"].values)
    ret_high = np.dot(w_high, high_b["ret"].values)

    dn_low.append({
        "month": ym,
        "ret_dn": ret_low
    })

    dn_mid.append({
        "month": ym,
        "ret_dn": ret_mid
    })

    dn_high.append({
        "month": ym,
        "ret_dn": ret_high
    })

/Users/brockwilliams/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Users/brockwilliams/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_slsqp_py.py:441: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
/Users/brockwilliams/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_slsqp_py.py:495: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
/Users/brockwilliams/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_slsqp_py.py:437: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/Users/brockwilliams/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_slsqp_py.py:441: RuntimeWarning: Values

In [7]:
low_opt  = pd.DataFrame(dn_low)
mid_opt  = pd.DataFrame(dn_mid)
high_opt = pd.DataFrame(dn_high)

low_opt["cum_dn"]  = (1 + low_opt["ret_dn"]).cumprod() - 1
mid_opt["cum_dn"]  = (1 + mid_opt["ret_dn"]).cumprod() - 1
high_opt["cum_dn"] = (1 + high_opt["ret_dn"]).cumprod() - 1

print("LOW DTS — Unhedged Duration-Neutral:")
print(low_opt.tail(), "\n")

print("MID DTS — Unhedged Duration-Neutral:")
print(mid_opt.tail(), "\n")

print("HIGH DTS — Unhedged Duration-Neutral:")
print(high_opt.tail(), "\n")

LOW DTS — Unhedged Duration-Neutral:
      month    ret_dn    cum_dn
27  2019-07  0.007519  0.022441
28  2019-08  0.003256  0.025770
29  2019-09  0.000715  0.026504
30  2019-10 -0.002795  0.023635
31  2019-11  0.000590  0.024239 

MID DTS — Unhedged Duration-Neutral:
      month    ret_dn    cum_dn
27  2019-07  0.016677  0.033497
28  2019-08  0.022194  0.056434
29  2019-09 -0.009693  0.046194
30  2019-10  0.001906  0.048188
31  2019-11  0.015104  0.064020 

HIGH DTS — Unhedged Duration-Neutral:
      month    ret_dn    cum_dn
27  2019-07  0.024659  0.050358
28  2019-08 -0.005868  0.044195
29  2019-09  0.006668  0.051157
30  2019-10  0.006516  0.058006
31  2019-11 -0.004154  0.053611 

